# NeRo-CLIP 사용법 — import해서 쓰기

CLIP은 부정(negation: *no / without / nothing …*)을 잘 못 읽습니다.
**NeRo-CLIP**은 얼어붙은(frozen) CLIP에 rank-8 residual 어댑터(8K 파라미터)를 붙여,
**부정 쿼리일 때만** 텍스트 임베딩을 교정합니다.

이 노트북은 학습된 어댑터를 **불러와서 쓰는 법**만 다룹니다.

> 저장소 루트에서 실행하세요.

In [ ]:
import sys
from pathlib import Path

# 저장소의 src/ 를 import 경로에 추가 (루트 또는 notebooks/ 에서 실행 가정)
for p in ["src", "../src"]:
    if (Path(p) / "nero.py").exists():
        sys.path.insert(0, p)
        break

import numpy as np
from clip_model import load_clip, encode_texts
from nero import route, NeRoAdapter, fetch_adapter
from utils import resolve_device

device = resolve_device()   # CUDA > MPS > CPU
print("device:", device)

## 1. CLIP + 학습된 어댑터 로드

어댑터는 NeRo-CLIP 저장소의 `pretrained/nero_lam0.75.pt` 에 들어 있습니다.
`fetch_adapter()` 가 저장소를 얕게(clone --depth 1) 받아 그 경로를 돌려줍니다.
(로컬에 `.pt` 가 있으면 그 경로를 바로 써도 됩니다.)

In [ ]:
model, processor = load_clip("openai/clip-vit-base-patch32", device)

# 방법 A) GitHub 저장소에서 사전학습 어댑터 가져오기
ckpt = fetch_adapter()                    # → artifacts/nero_repo/pretrained/nero_lam0.75.pt
# 방법 B) 로컬 체크포인트: ckpt = "artifacts/nero_lam0.75.pt"

adapter = NeRoAdapter.load(ckpt)
print(f"adapter loaded: rank={adapter.rank}, dim={adapter.dim}")

## 2. 라우터 — 언제 어댑터를 적용하나

11개 부정 표현(`no, not, without, nothing, never, neither, none, empty, absent, missing, n't`)을
정규식으로 감지합니다. 부정이 있으면 어댑터를 적용하고, 없으면 CLIP 원본을 그대로 씁니다.

In [ ]:
for q in ["a beach with no people",
          "a dog on a leash",
          "a man without a hat",
          "two people smiling"]:
    negated, markers = route(q)
    tag = f"ROUTE  markers={markers}" if negated else "skip"
    print(f"{tag:34s} | {q}")

## 3. 어댑터 적용 (핵심 recipe)

부정 쿼리는 **un-normalized** 임베딩을 어댑터에 넣고(논문과 동일), 결과를 L2 정규화합니다.
긍정 쿼리는 CLIP 원본을 그대로 둡니다.

In [ ]:
def nero_embed(query: str) -> np.ndarray:
    """NeRo-CLIP 텍스트 임베딩: 부정이면 어댑터로 교정, 아니면 CLIP 원본."""
    negated, _ = route(query)
    z = encode_texts(model, processor, [query], 1, device, normalize=not negated)[0]
    if negated:
        z = adapter.apply(z)
        z = z / (np.linalg.norm(z) + 1e-12)
    return z


q = "a beach with no people"
z_clip = encode_texts(model, processor, [q], 1, device)[0]   # 순수 CLIP
z_nero = nero_embed(q)                                       # NeRo 교정
print(f"부정 쿼리 '{q}': |Δ| = {np.linalg.norm(z_nero - z_clip):.3f}  (교정됨)")

qa = "a beach full of people"
za_clip = encode_texts(model, processor, [qa], 1, device)[0]
print(f"긍정 쿼리 '{qa}': |Δ| = {np.linalg.norm(nero_embed(qa) - za_clip):.3f}  (그대로)")

## 4. (옵션) 이미지 검색에 적용

Flickr30k 인덱스를 만들어 뒀다면 같은 쿼리로 CLIP vs NeRo top-k를 비교할 수 있습니다.

In [ ]:
from index_io import load_index
from retrieval import topk_search

idx_dir = Path("artifacts/flickr30k_clip")
if idx_dir.exists():
    image_emb, names, *_ = load_index(idx_dir)

    def search(query, k=5, use_nero=False):
        z = nero_embed(query) if use_nero else encode_texts(model, processor, [query], 1, device)[0]
        i, s = topk_search(z[None, :], image_emb, k)
        return [(names[int(j)], round(float(sc), 3)) for j, sc in zip(i[0], s[0])]

    q = "a dog without a leash"
    print("CLIP:", search(q)[:3])
    print("NeRo:", search(q, use_nero=True)[:3])
else:
    print("인덱스가 없습니다. 먼저 아래를 실행하세요:")
    print("  python src/data.py build-index --images-dir data/flickr30k/Images \\")
    print("    --captions-file data/flickr30k/captions.txt --output-dir artifacts/flickr30k_clip")

## 참고 — 재현 성능

held-out COCO MCQ-Neg (4지선다, 랜덤 25%): **CLIP 39.66 → NeRo 54.99**
(논문 open_clip 결과 54.70). 일반 COCO 검색은 30.36 → 30.37 로 보존됩니다.
어댑터 학습 코드를 포함한 논문 저장소는
<https://github.com/Algorythmsz/261RCOSE46101> 입니다.